# 10 - Apply YOLO Crop to All Images
Dùng YOLOv8n đã train để crop toàn bộ dataset → chuẩn bị cho Phase 3 (train MobileNet)

**v2: Dùng Letterbox resize thay vì Stretch resize để giữ tỉ lệ ảnh**

In [13]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO
from tqdm import tqdm

In [14]:
# ── Config ────────────────────────────────────────────────────────────
YOLO_MODEL_PATH  = "../models/yolov8n_rice_leaf.pt"
SOURCE_DIR       = "../data/augmented"   # Ảnh gốc đã augment (có class folders)
CROPPED_DIR      = "../data/cropped"     # Output: ảnh đã crop

CONF_THRESHOLD   = 0.3    # Confidence tối thiểu
PADDING          = 10     # Pixel padding xung quanh box
CROP_SIZE        = 224    # Output size cho MobileNet
FALLBACK_RESIZE  = True   # Nếu YOLO không detect → dùng ảnh gốc

# ┌─────────────────────────────────────────────────┐
# │  RESIZE_MODE = "letterbox" (giữ tỉ lệ + pad)   │  ← Khuyến nghị
# │  RESIZE_MODE = "stretch"   (kéo thẳng 224x224) │  ← Cũ
# └─────────────────────────────────────────────────┘
RESIZE_MODE      = "letterbox"
PAD_COLOR        = (0, 0, 0)   # Màu padding: đen (0,0,0) hoặc xám (114,114,114)

CLASS_NAMES = [
    "bacterial_blight",
    "barrow_brown_leaf_spot",
    "brown_spot",
    "healthy",
    "leaf_blast",
    "leaf_scald",
    "leaf_smut",
    "neck_blast",
    "rice_hispa",
    "sheath_blight",
    "tungro"
]

print(f"Resize mode : {RESIZE_MODE}")
print(f"Output size : {CROP_SIZE}x{CROP_SIZE}")

Resize mode : letterbox
Output size : 224x224


In [15]:
# ── 1. Load YOLO ──────────────────────────────────────────────────────
yolo = YOLO(YOLO_MODEL_PATH)
print(f"Loaded YOLO model: {YOLO_MODEL_PATH}")

Loaded YOLO model: ../models/yolov8n_rice_leaf.pt


In [16]:
# ── 2. Hàm resize ─────────────────────────────────────────────────────

def letterbox_resize(img_bgr, size=224, pad_color=(0, 0, 0)):
    """
    Resize giữ nguyên tỉ lệ width:height, pad thêm màu hai bên/trên dưới
    để đạt kích thước size x size.

    Ví dụ: crop 224x85
      → scale theo cạnh dài (224): new_w=224, new_h=85
      → pad top/bottom: (224-85)//2 = 69px mỗi bên
      → output 224x224, lá không bị méo
    """
    h, w   = img_bgr.shape[:2]
    scale  = size / max(h, w)
    new_w  = int(w * scale)
    new_h  = int(h * scale)

    resized  = cv2.resize(img_bgr, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    canvas   = np.full((size, size, 3), pad_color, dtype=np.uint8)
    pad_top  = (size - new_h) // 2
    pad_left = (size - new_w) // 2
    canvas[pad_top:pad_top + new_h, pad_left:pad_left + new_w] = resized
    return canvas


def smart_resize(img_bgr, size, mode, pad_color):
    """Wrapper: chọn letterbox hoặc stretch theo RESIZE_MODE."""
    if mode == "letterbox":
        return letterbox_resize(img_bgr, size, pad_color)
    else:
        return cv2.resize(img_bgr, (size, size))


def crop_largest_detection(img_bgr, results, padding=10):
    """
    Crop bbox lớn nhất từ kết quả YOLO.
    Returns: cropped image (BGR) hoặc None nếu không detect được.
    """
    h, w  = img_bgr.shape[:2]
    boxes = results[0].boxes

    if boxes is None or len(boxes) == 0:
        return None

    xyxy     = boxes.xyxy.cpu().numpy()
    areas    = (xyxy[:, 2] - xyxy[:, 0]) * (xyxy[:, 3] - xyxy[:, 1])
    best_idx = np.argmax(areas)

    x1, y1, x2, y2 = xyxy[best_idx].astype(int)
    x1 = max(0, x1 - padding)
    y1 = max(0, y1 - padding)
    x2 = min(w, x2 + padding)
    y2 = min(h, y2 + padding)

    crop = img_bgr[y1:y2, x1:x2]
    return crop if crop.size > 0 else None


def process_image(img_path, yolo_model, conf, padding,
                  crop_size, fallback, resize_mode, pad_color):
    """
    Returns (final_img_bgr, status)
    status: 'detected' | 'fallback' | 'error'
    """
    img = cv2.imread(str(img_path))
    if img is None:
        return None, 'error'

    results = yolo_model.predict(img, conf=conf, verbose=False)
    crop    = crop_largest_detection(img, results, padding)

    if crop is not None:
        out = smart_resize(crop, crop_size, resize_mode, pad_color)
        return out, 'detected'

    if fallback:
        out = smart_resize(img, crop_size, resize_mode, pad_color)
        return out, 'fallback'

    return None, 'no_detection'


print("Helper functions defined")
print(f"  Resize mode : {RESIZE_MODE}")

Helper functions defined
  Resize mode : letterbox


In [ ]:
# ── 3. Crop toàn bộ dataset ───────────────────────────────────────────
#
# Nếu đã chạy lần trước với stretch mode:
#   Đặt FORCE_REWRITE = True để ghi đè toàn bộ ảnh cũ.
#   Nếu chỉ crop ảnh chưa có, để False.
#
FORCE_REWRITE = True   # True = ghi đè (cần thiết khi đổi stretch -> letterbox)

stats = {cls: {'detected': 0, 'fallback': 0, 'error': 0} for cls in CLASS_NAMES}

for class_name in CLASS_NAMES:
    src_class_dir = os.path.join(SOURCE_DIR, class_name)
    dst_class_dir = os.path.join(CROPPED_DIR, class_name)

    if not os.path.isdir(src_class_dir):
        print(f"Skipping {class_name} (folder not found)")
        continue

    os.makedirs(dst_class_dir, exist_ok=True)

    images = [f for f in os.listdir(src_class_dir)
              if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    for img_file in tqdm(images, desc=f"{class_name}", ncols=80):
        src_path = os.path.join(src_class_dir, img_file)
        dst_path = os.path.join(dst_class_dir, img_file)

        if os.path.exists(dst_path) and not FORCE_REWRITE:
            stats[class_name]['detected'] += 1
            continue

        crop, status = process_image(
            src_path, yolo,
            CONF_THRESHOLD, PADDING, CROP_SIZE,
            FALLBACK_RESIZE, RESIZE_MODE, PAD_COLOR
        )

        if crop is not None:
            cv2.imwrite(dst_path, crop)

        stats[class_name][status] += 1

print("\nCropping done!")

bacterial_blight:  28%|█████▊               | 1117/4000 [01:12<03:04, 15.65it/s]

In [ ]:
# ── 4. Thống kê kết quả ───────────────────────────────────────────────
print("\n" + "="*75)
print(f"{'CLASS':<25} {'DETECTED':>10} {'FALLBACK':>10} {'ERROR':>10} {'TOTAL':>10}")
print("="*75)

total_det = total_fb = total_err = 0
for cls in CLASS_NAMES:
    d  = stats[cls]['detected']
    fb = stats[cls]['fallback']
    er = stats[cls]['error']
    total_det += d; total_fb += fb; total_err += er
    print(f"{cls:<25} {d:>10} {fb:>10} {er:>10} {d+fb+er:>10}")

print("="*75)
total = total_det + total_fb + total_err
print(f"{'TOTAL':<25} {total_det:>10} {total_fb:>10} {total_err:>10} {total:>10}")
print(f"\nDetection rate : {total_det/total*100:.1f}%")
print(f"Fallback rate  : {total_fb/total*100:.1f}%")
print(f"Resize mode    : {RESIZE_MODE}")

In [ ]:
# ── 5. Visualize so sanh stretch vs letterbox (4 anh mau) ─────────────
import random
random.seed(7)

sample_pairs = []
for cls in random.sample(CLASS_NAMES, 4):
    src_dir = os.path.join(SOURCE_DIR, cls)
    imgs = [f for f in os.listdir(src_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if imgs:
        sample_pairs.append((cls, random.choice(imgs)))

fig, axes = plt.subplots(len(sample_pairs), 3,
                          figsize=(12, len(sample_pairs) * 3.2))

for row, (cls, img_file) in enumerate(sample_pairs):
    src_path = os.path.join(SOURCE_DIR, cls, img_file)
    img      = cv2.imread(src_path)

    results = yolo.predict(img, conf=CONF_THRESHOLD, verbose=False)
    crop    = crop_largest_detection(img, results, PADDING)
    if crop is None:
        crop = img

    ch, cw  = crop.shape[:2]
    ratio   = cw / ch
    stretch = max(ratio, 1 / ratio)

    raw_rgb     = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    stretch_rgb = cv2.cvtColor(cv2.resize(crop, (224, 224)), cv2.COLOR_BGR2RGB)
    letter_rgb  = cv2.cvtColor(letterbox_resize(crop, 224, PAD_COLOR), cv2.COLOR_BGR2RGB)

    axes[row, 0].imshow(raw_rgb)
    axes[row, 0].set_title(f"[A] Raw crop ({cw}x{ch})\nratio={ratio:.2f}", fontsize=8)
    axes[row, 0].axis('off')

    c = 'red' if stretch >= 2.5 else ('orange' if stretch >= 1.5 else 'green')
    axes[row, 1].imshow(stretch_rgb)
    axes[row, 1].set_title(f"[B] Stretch 224x224\nstretch={stretch:.1f}x",
                            fontsize=8, color=c)
    axes[row, 1].axis('off')

    axes[row, 2].imshow(letter_rgb)
    axes[row, 2].set_title("[C] Letterbox 224x224\nti le giu nguyen",
                            fontsize=8, color='green')
    axes[row, 2].axis('off')
    axes[row, 0].set_ylabel(cls, fontsize=8)

plt.suptitle('Stretch vs Letterbox — [A] Raw  [B] Stretch (cu)  [C] Letterbox (moi)',
             fontsize=10)
plt.tight_layout()
plt.savefig('stretch_vs_letterbox_sample.png', dpi=100, bbox_inches='tight')
plt.show()
print("Saved -> stretch_vs_letterbox_sample.png")

In [ ]:
# ── 6. Visualize 12 anh crop final ────────────────────────────────────
random.seed(7)

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
axes = axes.flatten()

sample_classes = random.sample(CLASS_NAMES, min(12, len(CLASS_NAMES)))

for ax, cls in zip(axes, sample_classes):
    dst_dir = os.path.join(CROPPED_DIR, cls)
    imgs = [f for f in os.listdir(dst_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if not imgs:
        ax.axis('off')
        continue

    img = cv2.imread(os.path.join(dst_dir, random.choice(imgs)))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(cls, fontsize=9)
    ax.axis('off')

plt.suptitle(f'Sample Cropped Images (224x224) — mode: {RESIZE_MODE}', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── 7. Summary ────────────────────────────────────────────────────────
print("="*60)
print("NEXT STEP: Phase 3 - Train MobileNet on Cropped Images")
print("="*60)
print(f"Resize mode used : {RESIZE_MODE}")
print(f"Output saved to  : {os.path.abspath(CROPPED_DIR)}")
print()
print("1. Run 02_prepare_data.ipynb:")
print("   -> USE_AUGMENTED_DATA = False")
print("   -> SOURCE_DIR = '../data/cropped'")
print("2. Run 03_train_model.ipynb")
print("3. Run 04_evaluate_model.ipynb")
print("="*60)